# Build Sequential Models

## Setup

In [ ]:
import torch


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

In [ ]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

## Prepare our data

In [ ]:
import os
import kagglehub
import pandas as pd
import datasets
from transformers import AutoTokenizer
import torch
torch.manual_seed(42)
from torch.utils.data import DataLoader

In [ ]:
data_path = kagglehub.dataset_download("julian3833/jigsaw-toxic-comment-classification-challenge",
                                        "train.csv",
                                        output_dir="../data"
                                        )

In [ ]:
dataset = datasets.load_dataset("csv", data_files=data_path)
dataset

In [ ]:
import re

# existing patterns
username_pattern = re.compile(r'\[\[user.*?\]\]|\[\[user talk.*?\]\]|user:\w+', re.IGNORECASE)
whitespace_pattern = re.compile(r'\s+')
url_pattern = re.compile(r'http\S+|www\.\S+')
repeat_char_pattern = re.compile(r'(.)\1{2,}')

# wiki markup patterns
wiki_link_pattern = re.compile(r'\[\[(?:[^\]|]*\|)?([^\]]+)\]\]')
wiki_template_pattern = re.compile(r'\{\{.*?\}\}')
wiki_bold_italic_pattern = re.compile(r"'{2,5}")
wiki_heading_pattern = re.compile(r'=+\s*(.*?)\s*=+')

def preprocess_batch(batch):
    cleaned_texts = []

    for text in batch["comment_text"]:
        text = text.lower()

        # remove usernames
        text = username_pattern.sub(" ", text)

        # replace URLs with a [URL] token
        text = url_pattern.sub("[URL]", text)

        # convert [[link|text]] -> text
        text = wiki_link_pattern.sub(r"\1", text)

        # remove templates {{...}}
        text = wiki_template_pattern.sub(" ", text)

        # remove bold/italic markup
        text = wiki_bold_italic_pattern.sub("", text)

        # remove headings
        text = wiki_heading_pattern.sub(r"\1", text)

        # normalize stretched characters
        text = repeat_char_pattern.sub(r"\1\1", text)

        # normalize whitespace
        text = whitespace_pattern.sub(" ", text).strip()

        cleaned_texts.append(text)

    batch["comment_text"] = cleaned_texts
    return batch

In [ ]:
dataset = dataset["train"].map(preprocess_batch, batched=True, num_proc=4)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer.add_special_tokens({"additional_special_tokens": ["[URL]"]})

def tokenize_batch(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=256,
    )


In [ ]:
dataset = dataset.map(
    tokenize_batch,
    batched=True,
    num_proc=4
)
dataset = dataset.remove_columns(["comment_text", "id"])

dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "toxic",
        "severe_toxic",
        "obscene",
        "threat",
        "insult",
        "identity_hate"
    ]
)

In [ ]:
from transformers import DataCollatorWithPadding


LABEL_COLUMNS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Create a data collator that handles padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def collate_fn(batch):
    # Extract text features for padding
    text_features = [{
        "input_ids": item["input_ids"],
        "attention_mask": item["attention_mask"]
    } for item in batch]
    
    # Pad the batch
    padded = data_collator(text_features)
    
    # Extract labels
    labels = torch.stack([
        torch.tensor([item[label] for label in LABEL_COLUMNS], dtype=torch.float32)
        for item in batch
    ])
    
    X = {
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"]
    }
    
    return X, labels

In [ ]:
split = dataset.train_test_split(test_size=0.2, seed=42)
train_set = split["train"]
test_valid_set = split["test"]
split = test_valid_set.train_test_split(test_size=0.5, seed=42)
valid_set = split["train"]
test_set = split["test"]

In [ ]:
print(f"Train Shape: {train_set.shape}")
print(f"Valid Shape: {valid_set.shape}")
print(f"Test Shape: {test_set.shape}")


In [ ]:
BATCH_SIZE = 128
VOCAB_SIZE = len(tokenizer)
print(f"Vocabulary size: {VOCAB_SIZE}")

In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
# Add after creating dataloaders
sample_batch = next(iter(train_loader))
X_sample, y_sample = sample_batch

print(f"Input shapes:")
print(f"  input_ids: {X_sample['input_ids'].shape}")
print(f"  attention_mask: {X_sample['attention_mask'].shape}")
print(f"  labels: {y_sample.shape}")
print(f"  labels dtype: {y_sample.dtype}")

assert X_sample['input_ids'].shape[0] == BATCH_SIZE
assert y_sample.shape == (BATCH_SIZE, 6)
assert y_sample.dtype == torch.float32

## Training & Evaluation functions

In [ ]:
import torchmetrics
from torchmetrics.classification import MultilabelAveragePrecision

def evaluate_tm(model, data_loader, metric):
    """Evaluate model on a dataset."""
    
    model.eval()
    metric.reset()

    with torch.no_grad():
        for X_batch, y_batch in data_loader:

            X_batch = {k: v.to(device) for k, v in X_batch.items()}
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            probs = torch.sigmoid(logits)

            metric.update(probs.detach(), y_batch.int())

    return metric.compute()


def train(
    model,
    optimizer,
    loss_fn,
    train_metric,
    val_metric,
    train_loader,
    valid_loader,
    n_epochs,
    patience=2,
    factor=0.5,
    max_grad_norm=1.0,
    checkpoint_path="best_model.pth",
):

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        patience=patience,
        factor=factor
    )

    history = {
        "train_losses": [],
        "train_metrics": [],
        "valid_metrics": [],
    }

    best_val_metric = 0.0

    for epoch in range(n_epochs):

        model.train()
        train_metric.reset()

        total_loss = 0.0

        for index, (X_batch, y_batch) in enumerate(train_loader):

            X_batch = {k: v.to(device) for k, v in X_batch.items()}
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = loss_fn(logits, y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            optimizer.step()

            total_loss += loss.item()

            probs = torch.sigmoid(logits)

            train_metric.update(
                probs.detach(),
                y_batch.int()
            )

            print(
                f"\rBatch {index+1}/{len(train_loader)} "
                f"loss={total_loss/(index+1):.4f}",
                end=""
            )

        # ----- Epoch metrics -----

        train_metric_value = train_metric.compute().item()
        val_metric_value = evaluate_tm(model, valid_loader, val_metric).item()

        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric_value)
        history["valid_metrics"].append(val_metric_value)

        scheduler.step(val_metric_value)

        if val_metric_value > best_val_metric:

            best_val_metric = val_metric_value

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_metric": val_metric_value,
                    "history": history,
                },
                checkpoint_path,
            )

        print(
            f"\rEpoch {epoch+1}/{n_epochs}, "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train metric: {train_metric_value:.2%}, "
            f"valid metric: {val_metric_value:.2%} "
            f"{'[BEST]' if val_metric_value == best_val_metric else ''}"
        )

    return history

## Try different models architectures

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

### Simple Stacked LSTM

#### Architecture
- Random Embeddings with dim=256
- 2 LSTM layers
- a Linear output layer
- Use pack_padded sequences to fully ignored padding tokens

#### train and evaluation properities:
- Binary Cross Entropy as loss function (`nn.BCEWithLogitsLoss()`)
- Macro PR-AUC as evaluation metric (`torchmetrics.classification.MultilabelAveragePrecision`)
- NAdam as optimizer
- gradient clipping with max_norm 1.0

In [ ]:
class SimpleStackedLSTMModel(nn.Module):
    def __init__(self, vocab_size=30522, embed_dim=256, n_layers=2,
                 hidden_dim=128, pad_id=0, drop_out=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=drop_out)
        self.output = nn.Linear(hidden_dim, 6)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)                      
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),      
                                      batch_first=True, enforce_sorted=False) 
        _outputs, (h_n, c_n) = self.lstm(packed) 
        return self.output(h_n[-1])

#### Train & Evaluate model

In [ ]:
# Initialize model and move to device
model = SimpleStackedLSTMModel(vocab_size=len(tokenizer)).to(device)

# Training configuration
n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=0.001)

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
history = train(
    model, 
    optimizer, 
    xentropy, 
    macro_pr_auc,
    macro_pr_auc,
    train_loader, 
    valid_loader, 
    n_epochs,
    max_grad_norm=1.0,
    checkpoint_path='simple_stacked_lstm_model_best.pth'
)